# Visual Grounding with Qwen3-VL-8B

Test notebook: loads a few example rows from a `*_predictions_reason.csv`,
runs Qwen3-VL to estimate bounding boxes for each reasoning sentence
**and** each diagnosis name, and visualises the results.

Parsing uses the same `parse_reason_response` from `parse.py` that the
Django annotation interface uses, so grounding targets are identical to
what annotators see.

In [1]:
import os, sys

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"

# ============ MODEL SWITCH ============
# Choose: "qwen3_8b" or "molmo2_7b"
GROUNDING_MODEL = "molmo2_7b"
# ======================================

sys.path.insert(0, os.path.join(PROJECT_ROOT, "viz_ground", GROUNDING_MODEL))

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from util import (
    load_model,
    parse_reasoning_sentences,
    ground_reasoning_sentences,
    grounding_results_to_json,
)

print(f"Using grounding model: {GROUNDING_MODEL}")

## 1. Load a sample CSV

In [2]:
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
IMAGES_DIR = os.path.join(RESULTS_DIR, "images")

csv_path = os.path.join(RESULTS_DIR, "gpt53_predictions_reason.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows")
df.head()

## 2. Inspect parsed targets (no model needed)

Uses the same `parse_reason_response` as the Django interface.
Each diagnosis produces a `"diagnosis"` target (the name) plus
one `"sentence"` target per reasoning sentence.

In [3]:
sample_row = df.iloc[0]
print("Raw reason_classify:")
print(sample_row["reason_classify"][:500])
print("\n--- Parsed (same logic as Django interface) ---")
parsed = parse_reasoning_sentences(sample_row["reason_classify"])
for i, p in enumerate(parsed):
    tag = f"[{p['type'].upper()}]"
    print(f"\n{tag} Diagnosis: {p['diagnosis']}")
    print(f"       Text: {p['text'][:120]}")

## 3. Load Qwen3-VL model

In [4]:
model, processor = load_model()

## 4. Run visual grounding on a single example

In [6]:
# Pick a row that has an image
test_rows = df[df["image_path"].notna() & (df["image_path"] != "None")].head(5)
if test_rows.empty:
    test_rows = df.head(5)

row = test_rows.iloc[0]
print(f"ID: {row['id']}")

# Resolve image path
img_path = str(row.get("image_path", ""))
if not os.path.isfile(img_path):
    img_path = os.path.join(IMAGES_DIR, os.path.basename(img_path))
if not os.path.isfile(img_path):
    img_path = os.path.join(IMAGES_DIR, f"{row['id']}.jpg")

print(f"Image: {img_path}  (exists={os.path.isfile(img_path)})")
image = Image.open(img_path).convert("RGB")
image

In [7]:
results = ground_reasoning_sentences(model, processor, image, row["reason_classify"])
print(json.dumps(results, indent=2))

## 5. Visualise boxes on image

In [8]:
COLORS = ["#FF4444", "#44BB44", "#4444FF", "#FFAA00", "#AA44FF"]

def show_grounding(image, results, title=""):
    """Draw bounding boxes on the image for each grounding result."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image)
    W, H = image.size

    for i, r in enumerate(results):
        box = r.get("box")
        if not box:
            continue
        color = COLORS[i % len(COLORS)]
        linestyle = "--" if r.get("type") == "diagnosis" else "-"
        rect = patches.Rectangle(
            (box["x"] * W, box["y"] * H),
            box["w"] * W, box["h"] * H,
            linewidth=2, edgecolor=color, facecolor="none",
            linestyle=linestyle,
        )
        ax.add_patch(rect)
        prefix = "[D] " if r.get("type") == "diagnosis" else ""
        label = prefix + (r.get("text", "")[:40])
        ax.text(
            box["x"] * W, box["y"] * H - 4,
            label, color="white", fontsize=8,
            bbox=dict(facecolor=color, alpha=0.7, pad=2),
        )

    ax.set_title(title, fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

show_grounding(image, results, title=f"ID: {row['id']}  |  GT: {row['ground_truth']}")

## 6. Batch: process a few more rows

In [9]:
N_EXAMPLES = 3

for idx in range(min(N_EXAMPLES, len(test_rows))):
    row = test_rows.iloc[idx]
    img_p = str(row.get("image_path", ""))
    if not os.path.isfile(img_p):
        img_p = os.path.join(IMAGES_DIR, os.path.basename(img_p))
    if not os.path.isfile(img_p):
        img_p = os.path.join(IMAGES_DIR, f"{row['id']}.jpg")
    if not os.path.isfile(img_p):
        print(f"[SKIP] {row['id']}: image not found")
        continue

    img = Image.open(img_p).convert("RGB")
    res = ground_reasoning_sentences(model, processor, img, row["reason_classify"])
    print(f"\n--- {row['id']} ---")
    print(grounding_results_to_json(res))
    show_grounding(img, res, title=f"{row['id']}  |  {row['ground_truth']}")

## 7. Compare grounding: Combined vs Dscope vs Clinical

For `combined` rows, the batch script (`run_viz_ground.py`) automatically
collects 3 sets of boxes: `viz_grounding` (from combined image),
`viz_grounding_dscope` (from dscope-only), `viz_grounding_clinical`
(from clinical-only).

Here we preview the difference interactively before launching the full job.

In [10]:
# Pick a combined row
combined_rows = df[df["image_mode"] == "combined"].head(5)
row = combined_rows.iloc[0]
print(f"ID: {row['id']}  |  GT: {row['ground_truth']}  |  y16: {row['y16']}")
print(f"Reasoning (first 200 chars): {row['reason_classify'][:200]}")

# Derive paths for all 3 image variants
combined_path = os.path.join(IMAGES_DIR, f"{row['id']}.jpg")
dscope_path = combined_path.replace("_combined.", "_dscope.")
clinical_path = combined_path.replace("_combined.", "_photo.")

print(f"\nCombined: {combined_path} (exists: {os.path.isfile(combined_path)})")
print(f"Dscope:   {dscope_path} (exists: {os.path.isfile(dscope_path)})")
print(f"Clinical: {clinical_path} (exists: {os.path.isfile(clinical_path)})")

img_combined = Image.open(combined_path).convert("RGB")
img_dscope = Image.open(dscope_path).convert("RGB")
img_clinical = Image.open(clinical_path).convert("RGB")

In [11]:
# Run grounding on all 3 image variants using the SAME reasoning text
reason_text = row["reason_classify"]

print("Grounding on combined image...")
results_combined = ground_reasoning_sentences(model, processor, img_combined, reason_text)

print("Grounding on dscope image...")
results_dscope = ground_reasoning_sentences(model, processor, img_dscope, reason_text)

print("Grounding on clinical image...")
results_clinical = ground_reasoning_sentences(model, processor, img_clinical, reason_text)

print(f"\nTargets grounded: {len(results_combined)}")

In [12]:
def show_grounding_comparison(images, results_list, titles, suptitle=""):
    """Show grounding boxes side-by-side for multiple image variants."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(8 * n, 8))
    if n == 1:
        axes = [axes]

    for ax, img, results, title in zip(axes, images, results_list, titles):
        ax.imshow(img)
        W, H = img.size

        for i, r in enumerate(results):
            box = r.get("box")
            if not box:
                continue
            color = COLORS[i % len(COLORS)]
            linestyle = "--" if r.get("type") == "diagnosis" else "-"
            rect = patches.Rectangle(
                (box["x"] * W, box["y"] * H),
                box["w"] * W, box["h"] * H,
                linewidth=2, edgecolor=color, facecolor="none",
                linestyle=linestyle,
            )
            ax.add_patch(rect)
            prefix = "[D] " if r.get("type") == "diagnosis" else ""
            label = prefix + (r.get("text", "")[:35])
            ax.text(
                box["x"] * W, box["y"] * H - 4,
                label, color="white", fontsize=7,
                bbox=dict(facecolor=color, alpha=0.7, pad=1),
            )

        ax.set_title(title, fontsize=11)
        ax.axis("off")

    if suptitle:
        fig.suptitle(suptitle, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


show_grounding_comparison(
    [img_combined, img_dscope, img_clinical],
    [results_combined, results_dscope, results_clinical],
    ["Combined", "Dscope only", "Clinical only"],
    suptitle=f"ID: {row['id']}  |  GT: {row['ground_truth']}",
)

In [13]:
# Run comparison on a few more combined examples
N_COMPARE = 3

for idx in range(1, min(N_COMPARE + 1, len(combined_rows))):
    row = combined_rows.iloc[idx]
    c_path = os.path.join(IMAGES_DIR, f"{row['id']}.jpg")
    d_path = c_path.replace("_combined.", "_dscope.")
    p_path = c_path.replace("_combined.", "_photo.")

    if not (os.path.isfile(c_path) and os.path.isfile(d_path) and os.path.isfile(p_path)):
        print(f"Skipping {row['id']} — missing image variant")
        continue

    ic = Image.open(c_path).convert("RGB")
    id_ = Image.open(d_path).convert("RGB")
    ip = Image.open(p_path).convert("RGB")

    reason = row["reason_classify"]
    rc = ground_reasoning_sentences(model, processor, ic, reason)
    rd = ground_reasoning_sentences(model, processor, id_, reason)
    rp = ground_reasoning_sentences(model, processor, ip, reason)

    show_grounding_comparison(
        [ic, id_, ip], [rc, rd, rp],
        ["Combined", "Dscope only", "Clinical only"],
        suptitle=f"ID: {row['id']}  |  GT: {row['ground_truth']}  |  y16: {row['y16']}",
    )